In [4]:
# Load env variables
import json

from dotenv import load_dotenv
load_dotenv()
# Create an API client
from anthropic import Anthropic
from IPython.display import Markdown

client=Anthropic()
model= "claude-haiku-4-5"

def add_user_message(messages, text):
    user_message = {
        "role": "user",
        "content": text
    }
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {
        "role": "assistant",
        "content": text
    }
    messages.append(assistant_message)    

def chat(messages, system=None, stop_sequences=[]):
    params = {   
        "model": model,
        "max_tokens": 1000,
        "messages": messages
    }
    if system:
        params["system"] = system
    if stop_sequences:
        params["stop_sequences"] = stop_sequences

    response = client.messages.create(**params)
    return response.content[0].text


def generate_dataset():
    prompt= """ Generate an evaluation dataset for a prompt evaluation. the dataset will be used to evaluate prompts that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects, each representing task that requires Python, JSON, or a Regex to complete.

    Example output:
    ```json
    [ 
    {
    "task": "Description of task",
    },
    ....additonal
    ]
    ```
* Focus on tasks that can be solved by writing a single Python function, a single JSON object or a single JSON regex
* Focus on tasks that do not require writing much code

Please generate 3 objects
"""

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages,"```json")
    text = chat(messages,stop_sequences=["```"])
    return json.loads(text)

dataset = generate_dataset()
print(dataset)

with open('dataset.json','w') as f:
    json.dump(dataset,f,indent=2)


def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
    Please solve the following task:
    {test_case["task"]}
"""

    messages = []
    add_user_message(messages,prompt)
    output = chat(messages)
    return output


def run_test_case(test_case):
    """Calls run_promt, then grades the result"""
    output = run_prompt(test_case)

    #TODO - Grading
    score = 10

    return {
        "output" : output,
        "test_case" : test_case,
        "score": score
    }

def run_eval(dataset):
    """Loads teh dataset and calls run_test_case with each case"""
    results = []

    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    return results


with open("dataset.json","r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

print(json.dumps(results, indent=2))



[{'task': "Parse an AWS S3 bucket policy JSON and extract all principal ARNs that have the 's3:GetObject' permission", 'language': 'Python', 'complexity': 'medium'}, {'task': 'Create a JSON CloudFormation template snippet that defines an IAM role with a trust relationship allowing EC2 instances to assume the role', 'language': 'JSON', 'complexity': 'medium'}, {'task': 'Write a regex pattern to validate and extract AWS access key IDs (format: AKIA followed by 16 uppercase alphanumeric characters) from a text file', 'language': 'Regex', 'complexity': 'easy'}]
[
  {
    "output": "# AWS S3 Bucket Policy Parser\n\nHere's a solution to extract all principal ARNs with 's3:GetObject' permission:\n\n```python\nimport json\nfrom typing import Set, Union, List\n\ndef extract_getobject_principals(policy: Union[str, dict]) -> Set[str]:\n    \"\"\"\n    Parse an AWS S3 bucket policy and extract all principal ARNs \n    that have the 's3:GetObject' permission.\n    \n    Args:\n        policy: Eithe